In [1]:
import gzip
import fitdecode

path = "../data/raw/activities/21032562215.fit.gz"

In [2]:
from collections import Counter

counts = Counter()

with gzip.open(path, "rb") as f:
    with fitdecode.FitReader(f) as fit:
        for frame in fit:
            if isinstance(frame, fitdecode.FitDataMessage):
                counts[frame.name] += 1

counts

Counter({'unknown_233': 2911,
         'gps_metadata': 2911,
         'record': 2886,
         'unknown_325': 293,
         'unknown_326': 74,
         'time_in_zone': 19,
         'device_info': 14,
         'lap': 9,
         'split': 9,
         'unknown_104': 9,
         'split_summary': 4,
         'unknown_327': 4,
         'event': 3,
         'unknown_147': 3,
         'unknown_113': 3,
         'unknown_288': 2,
         'file_id': 1,
         'file_creator': 1,
         'activity': 1,
         'unknown_140': 1,
         'session': 1,
         'timestamp_correlation': 1,
         'unknown_22': 1,
         'unknown_141': 1,
         'unknown_394': 1,
         'device_settings': 1,
         'user_profile': 1,
         'unknown_79': 1,
         'sport': 1,
         'training_settings': 1,
         'zones_target': 1,
         'unknown_499': 1})

In [3]:
n = 0

with gzip.open(path, "rb") as f:
    with fitdecode.FitReader(f) as fit:
        for frame in fit:
            if isinstance(frame, fitdecode.FitDataMessage) and frame.name == "record":
                n += 1
                if n == 600:
                    for field in frame.fields:
                        print(field.name, "|", field.value, "|", field.units)
                    break

timestamp | 2026-08-25 17:16:00+00:00 | None
position_lat | 421490848 | semicircles
position_long | 397720728 | semicircles
distance | 1793.45 | m
accumulated_power | 175404 | watts
enhanced_speed | 2.902 | m/s
enhanced_altitude | 34.200000000000045 | m
unknown_140 | 2933 | None
power | 321 | watts
vertical_oscillation | 94.1 | mm
stance_time | 280.0 | ms
vertical_ratio | 8.75 | percent
step_length | 1074.0 | mm
cycle_length16 | 140.56 | m
heart_rate | 134 | bpm
cadence | 81 | rpm
activity_type | running | None
fractional_cadence | 0.5 | rpm
unknown_107 | 1 | None
unknown_135 | 28 | None
unknown_136 | 135 | None
unknown_143 | 22 | None
unknown_144 | 134 | None


In [4]:
import pandas as pd

rows = []

with gzip.open(path, "rb") as f:
    with fitdecode.FitReader(f) as fit:
        for frame in fit:
            if isinstance(frame, fitdecode.FitDataMessage) and frame.name == "record":
                rows.append({
                    "timestamp": frame.get_value("timestamp", fallback=None),
                    "distance_m": frame.get_value("distance", fallback=None),
                    "speed_ms": frame.get_value("enhanced_speed", fallback=None),
                    "altitude_m": frame.get_value("enhanced_altitude", fallback=None),
                    "heart_rate": frame.get_value("heart_rate", fallback=None),
                    "cadence": frame.get_value("cadence", fallback=None),
                    "fractional_cadence": frame.get_value("fractional_cadence", fallback=None),
                    "garmin_step_length_mm": frame.get_value("step_length", fallback=None),
                })

df = pd.DataFrame(rows)
df.head(10)

,timestamp,distance_m,speed_ms,altitude_m,heart_rate,cadence,fractional_cadence,garmin_step_length_mm
0,2026-08-25 17:06:01+00:00,1.11,1.493,96.4,88,48,0.5,933.0
1,2026-08-25 17:06:02+00:00,2.78,1.400,96.4,88,48,0.5,874.0
2,2026-08-25 17:06:03+00:00,5.24,1.400,96.2,88,48,0.5,874.0
3,2026-08-25 17:06:04+00:00,7.86,1.568,96.2,88,48,0.5,979.0
4,2026-08-25 17:06:05+00:00,11.18,2.025,96.0,88,48,0.5,1265.0
5,2026-08-25 17:06:06+00:00,14.31,2.435,96.0,88,87,0.0,1522.0
6,2026-08-25 17:06:07+00:00,18.32,3.014,95.8,89,87,0.0,1039.0
7,2026-08-25 17:06:08+00:00,21.78,3.023,95.8,89,87,0.0,1042.0
8,2026-08-25 17:06:09+00:00,25.13,3.415,95.6,89,87,0.0,1177.0
9,2026-08-25 17:06:10+00:00,28.46,3.415,95.6,90,86,0.5,1191.0


In [5]:
print(len(df))
df.describe()

2886


,distance_m,speed_ms,altitude_m,heart_rate,cadence,fractional_cadence,garmin_step_length_mm
count,2886.000000,2886.000000,2886.000000,2886.000000,2886.00000,2886.000000,2881.000000
mean,4148.169109,2.785077,48.693347,140.367637,80.74255,0.441788,1028.250607
std,2336.255781,0.276473,17.672693,7.835637,5.66248,0.160394,77.309320
min,1.110000,1.400000,30.800000,88.000000,0.00000,0.000000,481.000000
25%,2149.295000,2.659000,37.200000,139.000000,81.00000,0.500000,979.000000
50%,4195.000000,2.837000,42.600000,141.000000,82.00000,0.500000,1035.000000
75%,6225.005000,2.958000,51.400000,145.000000,82.00000,0.500000,1075.000000
max,8035.150000,3.835000,98.200000,153.000000,107.00000,0.500000,1522.000000


In [6]:
df["elapsed_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

In [7]:
df.loc[df["garmin_step_length_mm"].isna(), ["elapsed_s", "speed_ms", "cadence"]]

,elapsed_s,speed_ms,cadence
1541,1541.0,2.594,0
1542,1542.0,2.594,0
1543,1543.0,2.165,0
1544,1544.0,2.165,0
1545,1545.0,2.221,85


In [8]:
df.loc[df["cadence"] == 0, ["elapsed_s", "distance_m", "speed_ms"]]

,elapsed_s,distance_m,speed_ms
1540,1540.0,4438.40,2.594
1541,1541.0,4440.30,2.594
1542,1542.0,4443.20,2.594
1543,1543.0,4446.69,2.165
1544,1544.0,4449.91,2.165


In [9]:
df.loc[df["cadence"] > 95, ["elapsed_s", "speed_ms", "cadence", "heart_rate"]]

,elapsed_s,speed_ms,cadence,heart_rate
2698,2698.0,1.708,105,132
2699,2699.0,1.717,107,132


In [10]:
odd = df[(df["cadence"] == 0) | (df["cadence"] > 95) | (df["garmin_step_length_mm"].isna())]
odd[["elapsed_s", "speed_ms", "cadence", "garmin_step_length_mm"]]

,elapsed_s,speed_ms,cadence,garmin_step_length_mm
1540,1540.0,2.594,0,960.0
1541,1541.0,2.594,0,NaN
1542,1542.0,2.594,0,NaN
1543,1543.0,2.165,0,NaN
1544,1544.0,2.165,0,NaN
1545,1545.0,2.221,85,NaN
2698,2698.0,1.708,105,483.0
2699,2699.0,1.717,107,481.0


In [11]:
df["elapsed_s"].diff().value_counts()

elapsed_s
1.0    2885
Name: count, dtype: int64

In [12]:
import sys
sys.path.append("..")

from src.gait.loader import load_fit

test = load_fit(path)
print(len(test), test.columns.tolist())
test.head()

2886 ['timestamp', 'distance_m', 'speed_ms', 'altitude_m', 'heart_rate', 'cadence', 'fractional_cadence', 'garmin_step_length_mm', 'activity_id']


,timestamp,distance_m,speed_ms,altitude_m,heart_rate,cadence,fractional_cadence,garmin_step_length_mm,activity_id
0,2026-08-25 17:06:01+00:00,1.11,1.493,96.4,88,48,0.5,933.0,21032562215
1,2026-08-25 17:06:02+00:00,2.78,1.400,96.4,88,48,0.5,874.0,21032562215
2,2026-08-25 17:06:03+00:00,5.24,1.400,96.2,88,48,0.5,874.0,21032562215
3,2026-08-25 17:06:04+00:00,7.86,1.568,96.2,88,48,0.5,979.0,21032562215
4,2026-08-25 17:06:05+00:00,11.18,2.025,96.0,88,48,0.5,1265.0,21032562215


In [13]:
old = load_fit("../data/raw/activities/5813563613.fit.gz")
print(len(old))
old[["speed_ms", "altitude_m", "cadence", "garmin_step_length_mm"]].describe()

173


,speed_ms,altitude_m,cadence
count,173.000000,173.000000,173.000000
mean,3.382671,83.812717,81.416185
std,0.564387,11.980646,11.201130
min,0.000000,68.600000,0.000000
25%,3.200000,76.200000,82.000000
50%,3.368000,80.000000,83.000000
75%,3.639000,87.600000,84.000000
max,4.217000,116.400000,88.000000
